In [1]:
#Num 1: h vector
import numpy as np
import math
class customFileTool:
    @staticmethod
    def openFile(filepath):
        #opens and reads csv
        file = open(filepath, "r")
        data = file.read().split("\n")
        return data
    @staticmethod
    def findIndexesByKeyTerms(data, keyTerms):
        #finds index of strings after key terms to before the next key term
        #ie: data = "abcdef" and keyTerms = ["ab", "d"]
        #returns [(2, 3), (4, 6)]
        indexes = []
        #starts at the end of first key term
        indexStart = data.index(keyTerms[0])+len(keyTerms[0])
        for key in range(1, len(keyTerms)):
            indexEnd = data.index(keyTerms[key])
            indexes.append((indexStart, indexEnd))
            indexStart = data.index(keyTerms[key]) + len(keyTerms[key])
        #ends at the end of the string
        indexes.append((indexStart,len(data)))
        return indexes
    @staticmethod
    def multiSubstring(stringVal, substringIndexes):
        #given a list of index pairs and a string, returns a list of the substrings of the index pairs
        result = []
        for indexPair in substringIndexes:
            start = indexPair[0]
            end = indexPair[1]
            result.append(stringVal[start:end])
        return result
    @staticmethod
    def castList(data, dataType):
        #casts a list to a certain datatType
        result = []
        for datum in data:
            result.append(dataType(datum))
        return result
    @staticmethod
    def extractVectors(rawData):
        #extracts time, position vectors, velocity vectors, and ranges from a JPL ephemeris 
        #param: a JPL ephemeris as a list of strings, each representing a row
        startMarker = "$$SOE"
        endMarker = "$$EOE"
        startIndex = rawData.index(startMarker)+1
        endIndex = rawData.index(endMarker)

        result = []
        #iterates from startMarker to endMarker, incrementing by 4 because each datapoint is 4 rows tall.
        for rowIndex in range(startIndex, endIndex, 4):
            #getting the raw values, as strings
            rawTime = rawData[rowIndex]
            rawPosition = rawData[rowIndex + 1]
            rawVelocity = rawData[rowIndex + 2]
            rawRange = rawData[rowIndex + 3]

            #markers to mark start of each variable
            posSplitters = [" X =", " Y =", " Z ="]
            velSplitters = [" VX=", " VY=", " VZ="]
            rangeSplitters = [" LT=", " RG=", " RR="]

            time = rawTime.split(" = ")
            time[0] = float(time[0])
            #converts row [string] to float list 
            #in AU 
            position = customFileTool.castList(customFileTool.multiSubstring(rawPosition, customFileTool.findIndexesByKeyTerms(rawPosition, posSplitters)), float)
            velocity = customFileTool.castList(customFileTool.multiSubstring(rawVelocity, customFileTool.findIndexesByKeyTerms(rawVelocity, velSplitters)), float)
            #in au/day, need to cast to au/gaussian day
            #1 gaussian days in days
            gaussDay = 2*np.pi/365.25
            velocity = [velocity[i]/gaussDay for i in range(len(velocity))]
            ranges = customFileTool.castList(customFileTool.multiSubstring(rawRange, customFileTool.findIndexesByKeyTerms(rawRange, rangeSplitters)), float)

            row = [time, np.array(position), np.array(velocity), ranges]
            result.append(row)
        return result
    @staticmethod
    def roundList(vals, precision):
        #rounds a given list type by applying np.round onto each element.
        #param precision: integer, representing nubmer of digits
        result = []

        if(type(precision) == list):
            raise Exception("roundList: Precision type list not implemented yet")
        else:
            for datum in vals:
                result.append(np.round(datum, precision))
        return result

    


    






ModuleNotFoundError: No module named 'numpy'

In [2]:
class odLibTools:
    def newtonRaphson(function, derivative, initial_guess, error_tolerance):
        #given M, given e
        #guessXi
        #function func
        #function deriv
        #print(inital_guess)
        fxi  = function(initial_guess)
        dxi = derivative(initial_guess)

        nextGuess = initial_guess - (fxi/dxi)

        if (np.abs(nextGuess-initial_guess) < error_tolerance):
            return initial_guess

        return odLibTools.newtonRaphson(function, derivative, nextGuess, error_tolerance)



    def rotateZ(vector, alpha):
        # rotates a 3d vector clockwise about the Z axis
        # params: Vector: 3d vector in form of np array, alpha: amount to rotate cw in radians
        # returns: rotated vector
        a = alpha
        s, c = np.sin, np.cos
        rotationMatrixZ = np.array([[c(a), -s(a), 0],
                                   [s(a),  c(a), 0],
                                   [0    , 0,    1]])
        return rotationMatrixZ@vector


    def rotateX(vector, beta):
        # rotates a 3d vector clockwise about the X axis
        # params: Vector: 3d vector in form of np array, beta: amount to rotate cw in radians
        # returns: rotated vector
        b = beta
        s, c = np.sin, np.cos
        rotationMatrixX = np.array([[1,     0,    0],
                                   [ 0,  c(b),-s(b)],
                                   [ 0,  s(b), c(b)]])
        return rotationMatrixX@vector



In [3]:
class orbitalDetermination:
    '''
    ---- David K's Orbital Determination class ----
    
    * must include odLibTools to run
    * recommend using customFileTool to open files
    
    calculateOrbitalElements() 
    - use after setting velocity position and time
    - calculates orbital elements necesscary for ephemeris
    '''
    
    CONST_K = 0.01720209894 
    
    
    def __init__(self):
        self.position = False
        self.velocity = False
        
        self.semiMajorAxis = False
        self.eccentricity = False
        self.inclination = False
        self.longitudeOfAscendingNode = False
        self.argumentOfPerihelion = False
        self.meanAnomaly = False
        self.eccentricAnomaly = False
        self.trueAnomaly = False
        self.timeOfData = False
        self.specificAngularMomentum = False
        self.angleU = False

        
        
    def getOrbitalElementsTable(self, inDeg = False):
        table = {
            "a" : self.semiMajorAxis,
            "e" : self.eccentricity,
            "i" : self.inclination,
            "w" : self.argumentOfPerihelion,
            "OM" : self.longitudeOfAscendingNode,
            "M" : self.meanAnomaly,
            "E" : self.eccentricAnomaly,
            "v" : self.trueAnomaly,
            "t" : self.timeOfData,
            "h" : self.specificAngularMomentum,
            "U" : self.angleU,
            "T" : self.timeOfPerihelionPassage
        }
        if inDeg:
            table["i"] = np.rad2deg(table["i"])
            table["w"] = np.rad2deg(table["w"])
            table["OM"] = np.rad2deg(table["OM"])
            table["M"] = np.rad2deg(table["M"])
            table["E"] = np.rad2deg(table["E"])
            table["U"] = np.rad2deg(table["U"])
            table["v"] = np.rad2deg(table["v"])
        
        return table
    
    def getShorthandTable():
        table = {
            "a": "Semi-major Axis",
            "e": "Eccentricity",
            "i" : "Inclination",
            "w" : "Argument of Perihelion",
            "OM" : "Longitude of Ascending Node",
            "M" : "Mean Anomaly",
            "E" : "Eccentric Anomaly",
            "v" : "True Anomaly",
            "t" : "Time of Data",
            "h" : "Specific Angular Momentum",
            "U" : "Angle U",
            "T" : "Time of Perihelion Passage"
        }
        return table
        
    
    def getOrbitalElement(shorthand, inDeg = False):
        shorthands = self.getOrbitalElementsTable(inDeg)
        return shorthands[shorthand]
        
    '''
    position, velocity and sun vector must be set as 
    numpy vectors
    '''    
    def setPosition(self, pos):
        self.position = pos
    def setVelocity(self, vel):
        self.velocity = vel
    def setTime(self, time):
        self.timeOfData = time
    def setObliquity(self, obliq):
        self.obliquity = obliq
    def setSunVector(self, sunVec):
        self.sunVector = sunVec
    
    
    
    def calculateSpecificAngularMomentum(self):
        #In AU * AU / Gaussian day
        #calculates specific angular momentum by crossing r and velocity vectors
        cross = np.cross(self.position,self.velocity)
        self.specificAngularMomentum = cross
        return self.specificAngularMomentum
    
    
    def sinCosToAngle(self, sinVal, cosVal):
        # given the sin and cos of an angle, getAngle() returns the angle 
        # param: sin from -1 to 1, cos from -1 to 1
        # returns: angle in radians

        degVal = np.arccos(cosVal)
        #correct for negatives/postive sin
        degVal = math.copysign(degVal,sinVal)
        #conv to positive
        degVal %= 2*np.pi
        return degVal


    def magnitudeVector(self, vector):
        #returns the magnitude of a vector represented by a numpy array
        total = sum(vector**2)
        return np.sqrt(total)

    def calculateSemiMajorAxis(self):
        #calculates length of semi major axis in AU
        #uses Vis-viva equation
        denom = (2/self.magnitudeVector(self.position)
                -self.magnitudeVector(self.velocity)**2/1) #mu = 1
        self.semiMajorAxis = 1/denom
        return self.semiMajorAxis

    def calculateEccentricity(self):
        #calculates eccentricity of orbit
        #from ellipses
        frac = (self.magnitudeVector(self.specificAngularMomentum)**2
               /self.semiMajorAxis)
        self.eccentricity = np.sqrt(1-frac)
        return self.eccentricity

    def calculateInclination(self):
        #calculates inclination in rads
        #calculates from projection of h 
        #angle between plane of orbit and plane of ecliptic
        hz = self.specificAngularMomentum[2]
        h = self.magnitudeVector(self.specificAngularMomentum)
        self.inclination = np.arccos(hz/h)
        return self.inclination

    def calculateLongAscendNode(self):
        #calculates longitude of ascending node in rads
        # calculate from h projection
        # angle between vernal equinox and line of nodes
        hx = self.specificAngularMomentum[0]
        hy = self.specificAngularMomentum[1]
        h = self.magnitudeVector(self.specificAngularMomentum)
        sinVal = hx/(h*np.sin(self.inclination))
        cosVal = -hy/(h*np.sin(self.inclination))
        self.longitudeOfAscendingNode = self.sinCosToAngle(sinVal,cosVal)
        return self.longitudeOfAscendingNode

    def calcAngleU(self):
        #calculates angle U in radians
        #needed to solve argument of perhelion
        x = self.position[0]
        y = self.position[1]
        z = self.position[2]
        r = self.magnitudeVector(self.position)
        sinVal = z/(r*np.sin(self.inclination))
        cosVal = (x*np.cos(self.longitudeOfAscendingNode) + y*np.sin(self.longitudeOfAscendingNode))/r
        self.angleU = self.sinCosToAngle(sinVal, cosVal)
        return self.angleU
        
    def calcTrueAnomaly(self):
        #calculates true anomaly in radians
        #needed to solve argument of perihelion
        #from definition of an ellipse
        #angle from perihelion line to sun-asteroid vector
        r = self.magnitudeVector(self.position)
        h = self.magnitudeVector(self.specificAngularMomentum)
        a = self.semiMajorAxis
        e = self.eccentricity
        cosVal = (a*(1-e**2)/r -1) / e
        sinVal = (a*(1-e**2)/(h*e)) * np.dot(self.position,self.velocity)/r
        self.trueAnomaly = self.sinCosToAngle(sinVal, cosVal)
        return  self.trueAnomaly
        

    def calcArgumentOfPerihelion(self):
        #calculates argument of perihelion in radians
        #uses U and true anomaly to calculate 
        self.argumentOfPerihelion = (self.angleU - self.trueAnomaly) % (np.pi*2)
        return self.argumentOfPerihelion
        
    def calculateEccentricAnomaly(self):
        #calculates eccentric anomaly in radians
        r = self.magnitudeVector(self.position)
        a = self.semiMajorAxis
        e = self.eccentricity
        cosVal = (a*e + r*np.cos(self.trueAnomaly))/a
        sinVal = (r*np.sin(self.trueAnomaly))/(a*np.sqrt(1-e**2))
        self.eccentricAnomaly =  self.sinCosToAngle(sinVal, cosVal)
        return self.eccentricAnomaly

    def calcMeanAnomaly(self):
        #given gaussian units
        #calculates mean Anomaly in radians
        self.meanAnomaly = self.eccentricAnomaly - self.eccentricity * np.sin(self.eccentricAnomaly)
        return self.meanAnomaly


    def calcTimeOfPerihelionPassage(self):
        #given gaussian units
        #calculates time of Perihelion Passage in Julian Days

        #constant k (represents sqrt(constant G * Mass of sun))
        k = 0.0172020989484
        n = k/ (self.semiMajorAxis**(3/2))
        self.timeOfPerihelionPassage = self.timeOfData - self.meanAnomaly/n
        return self.timeOfPerihelionPassage
    
    def calculateOrbitalElements(self):
        #calculates all orbital elements given
        #must set position, velocity, and time first
        self.calculateSpecificAngularMomentum()
        self.calculateSemiMajorAxis()
        self.calculateEccentricity()
        self.calculateInclination()
        self.calculateLongAscendNode()
        self.calcAngleU()
        self.calcTrueAnomaly()
        self.calcArgumentOfPerihelion()
        self.calculateEccentricAnomaly()
        self.calcMeanAnomaly()
        self.calcTimeOfPerihelionPassage()
        
    #### Generating Ephemeris 😱😱😱😱😱😱 
    
    
    def generateEphemerisAtTime(self, time):
        #generates Right ascension and declination at time
        #returns (ra, dec) in radians
        #note: must run calculateOrbitalElements() first. 
        precision = 1e-6
        
        #shorthands
        k = self.CONST_K
        a = self.semiMajorAxis
        M = self.meanAnomaly
        e = self.eccentricity
        w = self.argumentOfPerihelion
        i = self.inclination
        om = self.longitudeOfAscendingNode
        #average rate of increase of the true anomaly 
        #(averaged over one peri|od P)
        n = k/(a**(3/2))
        #M_new is the new mean anomaly
        M_new = n*(time - self.timeOfData) + M
        #2 calculate E from M 
        #E_new is the new eccentric anomaly    
        #function: M = E-esin(E)
        #becomes f = M - (E - e*np.sin(E))
        func = (lambda E: M - (E - e*np.sin(E)))
        #derivative: -1 + ecosE
        deriv = (lambda E: -1 + e*np.cos(E))
        #use newtonRaphson to solve
        eccentricAnom = odLibTools.newtonRaphson(func, deriv,M, precision)
        E_new = eccentricAnom #shorthand
        #3. get cartesian coordinates of position vector
        x = a*np.cos(E_new) - a*e
        y = a*np.sqrt(1-e**2)*np.sin(E_new)
        z = 0
        #vector describing the asteroid position in orbit coordinates with the x axis coinciding with the perihelio
        pos_new = np.array([x,y,z])

        #rotate into ecliptic
        #First rotation is by (-ω), then by (i) and then by (Ω) to get the r vector in ecliptic coordinates
        pos_new = odLibTools.rotateZ(pos_new, w)
        pos_new = odLibTools.rotateX(pos_new, i)
        pos_new = odLibTools.rotateZ(pos_new, om)
        
        #rotate into equatorial
        pos_new = odLibTools.rotateX(pos_new, self.obliquity)
        #calculate range vector (rho)
        rho = pos_new + self.sunVector
        rhoHat = rho/self.magnitudeVector(rho)
        dec = np.arcsin(rhoHat[2])
        cosRA = rhoHat[0]/np.cos(dec)
        sinRA = rhoHat[1]/np.cos(dec)
        ra = self.sinCosToAngle(sinRA, cosRA)
        return(ra, dec)
        

In [4]:
#extract data:

rawData = customFileTool.openFile("/home/kimd/Downloads/KimInput.txt")
jul14midnightUTC = customFileTool.extractVectors(rawData)[24]
position = np.array(jul14midnightUTC[1])
velocity = np.array(jul14midnightUTC[2])
time = jul14midnightUTC[0][0]

#solve for orbital elements
od = orbitalDetermination()
#need to input
od.setVelocity(velocity)
od.setPosition(position)
od.setTime(time)

od.calculateOrbitalElements()
#printing results
odElementsTable = od.getOrbitalElementsTable(True)
for i in odElementsTable:
    print(i, odElementsTable[i])
print()

#need to input
obliq = np.deg2rad(23.4392911112)
od.setObliquity(obliq)

jul14SunVec = np.array([-3.689266984283532E-01, 9.472390240964750E-01, -6.56184E-05])
od.setSunVector(jul14SunVec)
july14 = 2458313.500


print("Ra and Declination in radians at", july14, "\n\t",od.generateEphemerisAtTime(july14))
ra,dec = od.generateEphemerisAtTime(july14)

print(np.rad2deg(ra), np.rad2deg(dec))

a 1.0567785034334412
e 0.34425536112950383
i 25.155251441985026
w 255.50235710782684
OM 236.23798039596574
M 140.42194699764906
E 150.2188255505139
v 158.95818225963967
t 2458313.5
h [-0.34107409  0.22800193  0.87362544]
U 54.460539367466524
T 2458158.722840479

Ra and Declination in radians at 2458313.5 
	 (4.789201242176811, -0.14035476151027243)
274.4010184155426 -8.041735469103822


In [5]:
print("😱")


😱


In [6]:

#checking percent errors
jplValues = {
    "a": 1.05671892483881,
    "e": .3442798363212599,
    "i": 25.15375982051783,
    "w": 255.5316184725226,
    "OM":236.2502480179119,
    "M":140.4194574969259,
    "T": 2458158.7208497203
}

shorthands = orbitalDetermination.getShorthandTable()
for i in jplValues:
    jplVal = jplValues[i]
    calcVal = odElementsTable[i]
    print("For: ",i, "\t["+shorthands[i]+"]")
    print("\tJpl Value:", jplVal)
    print("\tCalculated Value:", calcVal)
    error = np.abs(calcVal-jplVal)/jplVal
    print("\tPercent error:", error)
    isOk = error < 0.0002 #max error is .02%
    if(not isOk):
        print("Warning: Value out of accepted range")
    print()

For:  a 	[Semi-major Axis]
	Jpl Value: 1.05671892483881
	Calculated Value: 1.0567785034334412
	Percent error: 5.638073969406264e-05

For:  e 	[Eccentricity]
	Jpl Value: 0.3442798363212599
	Calculated Value: 0.34425536112950383
	Percent error: 7.109098231718857e-05

For:  i 	[Inclination]
	Jpl Value: 25.15375982051783
	Calculated Value: 25.155251441985026
	Percent error: 5.9300139535359e-05

For:  w 	[Argument of Perihelion]
	Jpl Value: 255.5316184725226
	Calculated Value: 255.50235710782684
	Percent error: 0.00011451171823934945

For:  OM 	[Longitude of Ascending Node]
	Jpl Value: 236.2502480179119
	Calculated Value: 236.23798039596574
	Percent error: 5.1926387587263666e-05

For:  M 	[Mean Anomaly]
	Jpl Value: 140.4194574969259
	Calculated Value: 140.42194699764906
	Percent error: 1.7729029634032664e-05

For:  T 	[Time of Perihelion Passage]
	Jpl Value: 2458158.7208497203
	Calculated Value: 2458158.722840479
	Percent error: 8.098577187194342e-10

